# **Assignment 05: MLLM**

**Available:** Sep 16, 2025 3:00pm until Sep 30, 2025 11:59pm

**Details**
- https://huggingface.co/datasets/AI4Math/MathVistaLinks to an external site.​
- Use test set​
- Add a lora to InternVL3 and SophiaVL-R1​
- Train both loras with testmini​
- Evaluate on test
- To get results on test set​, you need to run the leaderboard, 
- instructions are here: https://mathvista.github.io/#leaderboard
- Insights on why either IVL or SVL is better in the above two runs​
- Reports, code, video and insights

## Setup

In [1]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Install all packages for from the requirements.txt
%pip install -r requirements.txt

import torch
import torch.nn as nn
import torchvision
import datasets
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline
import time
import psutil
import gc
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
from collections import defaultdict
import numpy as np

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"Successfully authenticated with Hugging Face!")
        print(f"Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None


# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Found Hugging Face token in environment variables
Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl
Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl


In [2]:
# GPU Setup 
# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> mps (Apple Silicon) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    elif device.type == 'mps':
        # MPS doesn't have direct memory monitoring like CUDA
        # Return 0 as a placeholder
        return 0
    return 0


=== GPU Diagnostics ===
PyTorch version: 2.8.0+cu128
CUDA available: True
MPS available: False
CUDA version: 12.8
Number of GPUs detected: 2

=== All Available GPUs ===
GPU 0:
  Name: NVIDIA GeForce RTX 4090 Laptop GPU
  Total Memory: 15.70 GB
  Multi-processor count: 76
  Compute Capability: 8.9

GPU 1:
  Name: NVIDIA RTX A6000
  Total Memory: 47.53 GB
  Multi-processor count: 84
  Compute Capability: 8.6

Using GPU 1: NVIDIA RTX A6000
Selected device: cuda:1


In [3]:
# Function to clear memory for both CUDA and MPS
def clear_memory():
    """Clear CPU and GPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print(f"Cleared memory. Current CPU memory usage: {get_memory_usage():.2f} MB, GPU memory usage: {get_gpu_memory_usage():.2f} MB")

## Data Preparation and Processing

In [4]:
# Import the Dataset
# Source: https://huggingface.co/datasets/AI4Math/MathVista

from datasets import load_dataset

dataset = load_dataset("AI4Math/MathVista")

In [5]:
# Examine the dataset structure
print("Dataset structure:")
print(dataset)
print("\nDataset keys:")
print(dataset.keys())

# Check the structure of each split
for split_name in dataset.keys():
    print(f"\n{split_name} split:")
    print(f"  Number of examples: {len(dataset[split_name])}")
    if len(dataset[split_name]) > 0:
        print(f"  Features: {dataset[split_name].features}")
        print(f"  First example keys: {list(dataset[split_name][0].keys())}")
        
        # Show a sample of the first example
        sample = dataset[split_name][0]
        print(f"  Sample data:")
        for key, value in sample.items():
            if key == 'image' and value is not None:
                print(f"    {key}: PIL Image ({value.size if hasattr(value, 'size') else 'unknown size'})")
            elif isinstance(value, str) and len(value) > 100:
                print(f"    {key}: {value[:100]}...")
            else:
                print(f"    {key}: {value}")
        break

Dataset structure:
DatasetDict({
    testmini: Dataset({
        features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
        num_rows: 5141
    })
})

Dataset keys:
dict_keys(['testmini', 'test'])

testmini split:
  Number of examples: 1000
  Features: {'pid': Value('string'), 'question': Value('string'), 'image': Value('string'), 'decoded_image': Image(mode=None, decode=True), 'choices': List(Value('string')), 'unit': Value('string'), 'precision': Value('float64'), 'answer': Value('string'), 'question_type': Value('string'), 'answer_type': Value('string'), 'metadata': {'category': Value('string'), 'context': Value('string'), 'grade': Value('string'), 'img_height': Value('int64'), 

In [6]:
# Data Processing Functions
import json
from typing import Dict, List, Any, Tuple
from PIL import Image
import io
import base64

class MathVistaProcessor:
    def __init__(self):
        self.processed_data = {
            'testmini': [],
            'test': []
        }
    
    def format_question_for_models(self, example: Dict) -> Tuple[str, Image.Image]:
        """Format question and image for model input"""
        # Get the image
        image = example['decoded_image']
        
        # Format the question based on question type
        question = example['question']
        query = example.get('query', '')
        
        # Add context based on answer type
        if example['answer_type'] == 'float':
            if example.get('precision'):
                formatted_question = f"{question}\n\nPlease provide your answer as a floating-point number with {int(example['precision'])} decimal place(s)."
            else:
                formatted_question = f"{question}\n\nPlease provide your answer as a floating-point number."
        elif example['answer_type'] == 'integer':
            formatted_question = f"{question}\n\nPlease provide your answer as an integer."
        elif example['question_type'] == 'multi_choice':
            if example.get('choices'):
                choices_text = '\n'.join([f"{i+1}. {choice}" for i, choice in enumerate(example['choices'])])
                formatted_question = f"{question}\n\nChoices:\n{choices_text}\n\nPlease select the correct answer."
            else:
                formatted_question = question
        else:
            formatted_question = question
        
        # Add any additional query information
        if query and query != question:
            formatted_question = f"{formatted_question}\n\nAdditional context: {query}"
        
        return formatted_question, image
    
    def process_split(self, split_name: str, max_samples: int = None) -> List[Dict]:
        """Process a specific split of the dataset"""
        split_data = dataset[split_name]
        processed_examples = []
        
        print(f"Processing {split_name} split...")
        
        # Limit samples if specified
        if max_samples:
            split_data = split_data.select(range(min(max_samples, len(split_data))))
        
        for idx, example in enumerate(tqdm(split_data, desc=f"Processing {split_name}")):
            try:
                # Format question and get image
                formatted_question, image = self.format_question_for_models(example)
                
                # Create processed example
                processed_example = {
                    'pid': example['pid'],
                    'formatted_question': formatted_question,
                    'original_question': example['question'],
                    'image': image,
                    'ground_truth_answer': example['answer'],
                    'question_type': example['question_type'],
                    'answer_type': example['answer_type'],
                    'choices': example.get('choices'),
                    'unit': example.get('unit'),
                    'precision': example.get('precision'),
                    'metadata': example['metadata'],
                    'query': example.get('query', ''),
                }
                
                processed_examples.append(processed_example)
                
                # Print first example for verification
                if idx == 0:
                    print(f"\nFirst example from {split_name}:")
                    print(f"PID: {processed_example['pid']}")
                    print(f"Question type: {processed_example['question_type']}")
                    print(f"Answer type: {processed_example['answer_type']}")
                    print(f"Formatted question: {processed_example['formatted_question'][:200]}...")
                    print(f"Ground truth: {processed_example['ground_truth_answer']}")
                    print(f"Image size: {processed_example['image'].size}")
                    print("-" * 50)
                
            except Exception as e:
                print(f"Error processing example {idx} in {split_name}: {e}")
                continue
        
        self.processed_data[split_name] = processed_examples
        print(f"Successfully processed {len(processed_examples)} examples from {split_name}")
        
        return processed_examples
    
    def get_statistics(self):
        """Get statistics about the processed data"""
        stats = {}
        
        for split_name, examples in self.processed_data.items():
            if not examples:
                continue
                
            stats[split_name] = {
                'total_examples': len(examples),
                'question_types': {},
                'answer_types': {},
                'categories': {},
            }
            
            for example in examples:
                # Question types
                q_type = example['question_type']
                stats[split_name]['question_types'][q_type] = stats[split_name]['question_types'].get(q_type, 0) + 1
                
                # Answer types
                a_type = example['answer_type']
                stats[split_name]['answer_types'][a_type] = stats[split_name]['answer_types'].get(a_type, 0) + 1
                
                # Categories
                category = example['metadata']['category']
                stats[split_name]['categories'][category] = stats[split_name]['categories'].get(category, 0) + 1
        
        return stats

# Initialize processor
processor = MathVistaProcessor()

print("MathVista processor initialized successfully!")

MathVista processor initialized successfully!


In [7]:
# Process the testmini split (for training LoRA)
print("Processing testmini split for training...")
testmini_processed = processor.process_split('testmini')

print(f"\nTestmini processing completed!")
print(f"Total examples processed: {len(testmini_processed)}")

# Show some statistics
testmini_stats = processor.get_statistics()
print("\nTestmini Statistics:")
print(f"Question types: {testmini_stats['testmini']['question_types']}")
print(f"Answer types: {testmini_stats['testmini']['answer_types']}")
print(f"Categories: {list(testmini_stats['testmini']['categories'].keys())}")

Processing testmini split for training...
Processing testmini split...


Processing testmini:   0%|          | 0/1000 [00:00<?, ?it/s]


First example from testmini:
PID: 1
Question type: free_form
Answer type: float
Formatted question: When a spring does work on an object, we cannot find the work by simply multiplying the spring force by the object's displacement. The reason is that there is no one value for the force-it changes. Ho...
Ground truth: 1.2
Image size: (1514, 720)
--------------------------------------------------
Successfully processed 1000 examples from testmini

Testmini processing completed!
Total examples processed: 1000

Testmini Statistics:
Question types: {'free_form': 460, 'multi_choice': 540}
Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
Categories: ['math-targeted-vqa', 'general-vqa']
Successfully processed 1000 examples from testmini

Testmini processing completed!
Total examples processed: 1000

Testmini Statistics:
Question types: {'free_form': 460, 'multi_choice': 540}
Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
Categories: ['math-targeted-vqa',

In [8]:
# Process the test split (for evaluation)
print("Processing test split for evaluation...")
test_processed = processor.process_split('test')

print(f"\nTest processing completed!")
print(f"Total examples processed: {len(test_processed)}")

# Show complete statistics
all_stats = processor.get_statistics()
print("\n" + "="*60)
print("COMPLETE DATASET STATISTICS")
print("="*60)

for split_name, stats in all_stats.items():
    print(f"\n{split_name.upper()} SPLIT:")
    print(f"  Total examples: {stats['total_examples']}")
    print(f"  Question types: {stats['question_types']}")
    print(f"  Answer types: {stats['answer_types']}")
    print(f"  Categories: {stats['categories']}")
    print("-" * 40)

Processing test split for evaluation...
Processing test split...


Processing test:   0%|          | 0/5141 [00:00<?, ?it/s]


First example from test:
PID: 1001
Question type: free_form
Answer type: integer
Formatted question: In how many years, is the percentage of labor tax greater than 3 %?

Please provide your answer as an integer.

Additional context: Hint: Please answer the question requiring an integer answer and pro...
Ground truth: 
Image size: (981, 650)
--------------------------------------------------
Successfully processed 5141 examples from test

Test processing completed!
Total examples processed: 5141

COMPLETE DATASET STATISTICS

TESTMINI SPLIT:
  Total examples: 1000
  Question types: {'free_form': 460, 'multi_choice': 540}
  Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
  Categories: {'math-targeted-vqa': 540, 'general-vqa': 460}
----------------------------------------

TEST SPLIT:
  Total examples: 5141
  Question types: {'free_form': 2289, 'multi_choice': 2852}
  Answer types: {'integer': 2043, 'text': 2852, 'float': 232, 'list': 14}
  Categories: {'general-vqa': 

In [9]:
# Utility functions to access processed data
def get_training_data():
    """Get processed testmini data for training LoRA"""
    return processor.processed_data['testmini']

def get_evaluation_data():
    """Get processed test data for evaluation"""
    return processor.processed_data['test']

def get_sample_batch(split='testmini', batch_size=4, start_idx=0):
    """Get a sample batch for testing model inference"""
    if split not in processor.processed_data:
        print(f"Split '{split}' not found. Available splits: {list(processor.processed_data.keys())}")
        return []
    
    data = processor.processed_data[split]
    end_idx = min(start_idx + batch_size, len(data))
    return data[start_idx:end_idx]

def save_processed_data(filename_prefix="mathvista_processed"):
    """Save processed data to files for later use"""
    import pickle
    
    # Save testmini data
    with open(f"data/{filename_prefix}_testmini.pkl", 'wb') as f:
        pickle.dump(processor.processed_data['testmini'], f)
    print(f"Testmini data saved to data/{filename_prefix}_testmini.pkl")
    
    # Save test data
    with open(f"data/{filename_prefix}_test.pkl", 'wb') as f:
        pickle.dump(processor.processed_data['test'], f)
    print(f"Test data saved to data/{filename_prefix}_test.pkl")
    
    # Save statistics
    stats = processor.get_statistics()
    with open(f"data/{filename_prefix}_stats.json", 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"Statistics saved to data/{filename_prefix}_stats.json")

def load_processed_data(filename_prefix="mathvista_processed"):
    """Load previously processed data"""
    import pickle
    
    try:
        # Load testmini data
        with open(f"data/{filename_prefix}_testmini.pkl", 'rb') as f:
            processor.processed_data['testmini'] = pickle.load(f)
        print(f"Testmini data loaded from data/{filename_prefix}_testmini.pkl")
        
        # Load test data
        with open(f"data/{filename_prefix}_test.pkl", 'rb') as f:
            processor.processed_data['test'] = pickle.load(f)
        print(f"Test data loaded from data/{filename_prefix}_test.pkl")
        
        return True
    except FileNotFoundError as e:
        print(f"Could not load processed data: {e}")
        return False

# Save the processed data
save_processed_data()

print("\n" + "="*60)
print("DATASET PROCESSING COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"✓ Testmini split: {len(processor.processed_data['testmini'])} examples (for LoRA training)")
print(f"✓ Test split: {len(processor.processed_data['test'])} examples (for evaluation)")
print("✓ Data formatted for both InternVL3 and SophiaVL-R1 models")
print("✓ Processed data saved to pickle files")
print("\nYou can now proceed with:")
print("1. Training LoRA adapters on both models using testmini data")
print("2. Evaluating both models on test data")
print("3. Comparing performance between InternVL3 and SophiaVL-R1")

Testmini data saved to data/mathvista_processed_testmini.pkl
Test data saved to data/mathvista_processed_test.pkl
Statistics saved to data/mathvista_processed_stats.json

DATASET PROCESSING COMPLETED SUCCESSFULLY!
✓ Testmini split: 1000 examples (for LoRA training)
✓ Test split: 5141 examples (for evaluation)
✓ Data formatted for both InternVL3 and SophiaVL-R1 models
✓ Processed data saved to pickle files

You can now proceed with:
1. Training LoRA adapters on both models using testmini data
2. Evaluating both models on test data
3. Comparing performance between InternVL3 and SophiaVL-R1
Test data saved to data/mathvista_processed_test.pkl
Statistics saved to data/mathvista_processed_stats.json

DATASET PROCESSING COMPLETED SUCCESSFULLY!
✓ Testmini split: 1000 examples (for LoRA training)
✓ Test split: 5141 examples (for evaluation)
✓ Data formatted for both InternVL3 and SophiaVL-R1 models
✓ Processed data saved to pickle files

You can now proceed with:
1. Training LoRA adapters on b

## Import InternVL3 Model

In [10]:
# Source: https://huggingface.co/OpenGVLab/InternVL3-78B

# Load model directly
from transformers import AutoModel
model = AutoModel.from_pretrained("OpenGVLab/InternVL3-78B", trust_remote_code=True, dtype="auto")



FlashAttention2 is not installed.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

In [ ]:
clear_memory

<function __main__.clear_memory()>

: 

## Import SophiaVL-R1 Model

In [5]:
## Source: https://huggingface.co/bunny127/SophiaVL-R1-Thinking-Reward-Model-3B

# Load model directly
from transformers import AutoProcessor, AutoModelForVision2Seq
import requests
from PIL import Image
import torch

# Load processor and model
processor = AutoProcessor.from_pretrained("bunny127/SophiaVL-R1-Thinking-Reward-Model-3B")
model = AutoModelForVision2Seq.from_pretrained("bunny127/SophiaVL-R1-Thinking-Reward-Model-3B")

# Move model to device
model = model.to(device)

# Download and resize image to reduce token count
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
image = Image.open(requests.get(image_url, stream=True).raw)

# Resize image to reduce token count (smaller image = fewer tokens)
max_size = 512  # Reduce from default to limit tokens
image = image.resize((max_size, int(max_size * image.height / image.width)), Image.Resampling.LANCZOS)

print(f"Resized image to: {image.size}")

messages = [
    {
        "role": "user", 
        "content": [
            {"type": "image", "image": image},  # Use PIL image directly
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]

# Process with truncation to handle long sequences
try:
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        max_length=9500,  # Leave some room for generation
        truncation=True,  # Truncate if too long
    ).to(model.device)
    
    print(f"Input sequence length: {inputs['input_ids'].shape[-1]} tokens")
    
    # Generate with shorter output to stay within limits
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=60,  # Reduced from 40
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id
        )
    
    # Decode only the generated part
    generated_text = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    print(f"Generated response: {generated_text}")
    
except Exception as e:
    print(f"Error during inference: {e}")
    print("The sequence is still too long even after resizing. Try with an even smaller image or shorter text.")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Resized image to: (512, 384)
Input sequence length: 280 tokens
Generated response: The candy in your hand has an animal design on it, but I can't identify the specific animal from the image provided. It appears to be a small, stylized figure that resembles an animal, possibly a bird or a fish, but without more details, I can't be certain.
